## ET Assignment 3

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso

| Month | TV Advertising | Radio Advertising | Digital Advertising | Sales |
| --- | --- | --- | --- | --- |
| M1 | 2 | 3 | 5 | 20 |
| M2 | 3 | 4 | 6 | 26 |
| M3 | 4 | 5 | 7 | 32 |
| M4 | 5 | 6 | 8 | 38 |
| M5 | 6 | 7 | 10 | 47 |
| M6 | 7 | 8 | 11 | 53 |
| M7 | 8 | 9 | 12 | 59 |
| M8 | 9 | 10 | 14 | 67 |
| M9 | 10 | 11 | 15 | 73 |
| M10 | 11 | 12 | 16 | 79 |
| M11 | 12 | 13 | 18 | 89 |
| M12 | 13 | 14 | 19 | 95 |

Suppose a company records advertising expenditure in three channels and observes monthly sales.

Target variable: **Sales**  
Features: **TV, Radio, Digital Advertising**  

The three advertising variables are intentionally highly correlated.

In [2]:
data = {
    "Month": [f"M{i}" for i in range(1, 13)],
    "TV Advertising": [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13],
    "Radio Advertising": [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    "Digital Advertising": [5, 6, 7, 8, 10, 11, 12, 14, 15, 16, 18, 19],
    "Sales": [20, 26, 32, 38, 47, 53, 59, 67, 73, 79, 89, 95]
}

df = pd.DataFrame(data)
df

,Month,TV Advertising,Radio Advertising,Digital Advertising,Sales
0,M1,2,3,5,20
1,M2,3,4,6,26
2,M3,4,5,7,32
3,M4,5,6,8,38
4,M5,6,7,10,47
5,M6,7,8,11,53
6,M7,8,9,12,59
7,M8,9,10,14,67
8,M9,10,11,15,73
9,M10,11,12,16,79


1) Calculate the correlation of TV Advertising with Sales. Calculate the correlation of Radio Advertising with Sales. Calculate the correlation of Digital Advertising with Sales.

In [3]:
features = ["TV Advertising", "Radio Advertising", "Digital Advertising"]

correlations = df[features].corrwith(df["Sales"])
correlations.to_frame(name="Correlation with Sales")

,Correlation with Sales
TV Advertising,0.999169
Radio Advertising,0.999169
Digital Advertising,0.999489


The correlations are all very close to 1, showing a very strong positive linear relationship between each advertising variable and Sales.

2) Apply Ridge Regression with α = 1. Compare effect of Ridge α = 0.1 and α = 10 and analyse how the coefficients change.

Before applying Ridge, the input features are standardized so that the regularization penalty is applied comparably to all three features.

In [4]:
X = df[features]
y = df["Sales"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=features)
X_scaled_df

,TV Advertising,Radio Advertising,Digital Advertising
0,-1.593255,-1.593255,-1.496157
1,-1.303572,-1.303572,-1.274504
2,-1.013890,-1.013890,-1.052851
3,-0.724207,-0.724207,-0.831198
4,-0.434524,-0.434524,-0.387893
5,-0.144841,-0.144841,-0.166240
6,0.144841,0.144841,0.055413
7,0.434524,0.434524,0.498719
8,0.724207,0.724207,0.720372
9,1.013890,1.013890,0.942025


In [5]:
ridge_results = []

for alpha in [0.1, 1, 10]:
    model = Ridge(alpha=alpha)
    model.fit(X_scaled, y)

    ridge_results.append({
        "Alpha": alpha,
        "TV Advertising": model.coef_[0],
        "Radio Advertising": model.coef_[1],
        "Digital Advertising": model.coef_[2],
        "Intercept": model.intercept_
    })

ridge_coefficients = pd.DataFrame(ridge_results).set_index("Alpha")
ridge_coefficients

,TV Advertising,Radio Advertising,Digital Advertising,Intercept
Alpha,,,,
0.1,7.160076,7.160076,9.321179,56.5
1.0,7.596884,7.596884,7.871133,56.5
10.0,6.174782,6.174782,6.199274,56.5


For Ridge, increasing α increases the regularization strength. Therefore, the coefficients are shrunk toward zero. At α = 0.1 the shrinkage is weakest, at α = 1 it is stronger, and at α = 10 it is strongest. Because TV and Radio are perfectly aligned in this dataset, Ridge keeps equal coefficients for those two variables rather than setting either one exactly to zero.

3) Apply Lasso with α = 1. Which feature is effectively removed from the model? Compare effect of Lasso α = 0.1 and α = 10 and analyse how the coefficients change.

In [6]:
lasso_results = []

for alpha in [0.1, 1, 10]:
    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X_scaled, y)

    lasso_results.append({
        "Alpha": alpha,
        "TV Advertising": model.coef_[0],
        "Radio Advertising": model.coef_[1],
        "Digital Advertising": model.coef_[2],
        "Intercept": model.intercept_
    })

lasso_coefficients = pd.DataFrame(lasso_results).set_index("Alpha")
lasso_coefficients

,TV Advertising,Radio Advertising,Digital Advertising,Intercept
Alpha,,,,
0.1,10.303323,4.810966e-16,13.307309,56.5
1.0,9.842355,4.292862e-14,12.867316,56.5
10.0,5.232677,3.315866e-14,8.467382,56.5


With Lasso, the **Radio Advertising** coefficient becomes zero, so Radio is effectively removed from the model. This is a characteristic difference from Ridge: Lasso can drive coefficients exactly to zero, performing feature selection. As α increases from 0.1 to 10, the non-zero coefficients are also pushed closer to zero.

### Comparison of Ridge and Lasso

| Method | Effect of increasing α | Feature selection |
| --- | --- | --- |
| Ridge | Coefficients shrink smoothly toward zero | No coefficient is forced exactly to zero here |
| Lasso | Coefficients shrink, with some reaching exactly zero | Yes; Radio Advertising is removed |